# Feature Transformations & Feature Scaling
## A visual, classroom-ready guide — from intuition to model-ready data

**Teaching goal:** By the end, a beginner should be able to explain:
1. What problem a transformation or scaler is trying to solve.
2. The mathematical formula in plain language.
3. What the before/after graphs prove — and what they do *not* prove.
4. Which method is a reasonable first choice in a particular scenario.
5. Why preprocessing must be fitted on training data only.

> **How to teach this notebook:** Read each “Say this to students” box, run the cell below it, and ask students to compare the graphs before reading the observation. Graphs are evidence of what happened to this example dataset, not proof that the method will improve every real model.

### Road map
- **Part A — Transformations:** Log, square root, exponential, Box–Cox, Yeo–Johnson / power transforms.
- **Part B — Scaling:** Standard, Min–Max, Robust, MaxAbs, and row-wise Normalization.
- **Part C — Decision guides:** choose by data problem, compare outcomes, avoid common mistakes, and prevent leakage.


## 0. Setup: libraries and our demonstration data

We create two kinds of data:
- A **right-skewed positive feature** (think income, property prices, or transaction amount).
- A **mixed-sign feature** containing negative, zero, and positive values.

The dataset is generated in this notebook, so no CSV download is needed.

**Say this to students:** “We are not changing data just to make a graph pretty. We first identify a problem, apply a suitable method, and then check whether the problem became smaller.”

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import boxcox, skew
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler,
    PowerTransformer, Normalizer
)
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(42)

# Positive, right-skewed data with a few large values
x = np.random.lognormal(mean=2.0, sigma=0.85, size=1200)
x[:5] = [100, 130, 160, 190, 250]

# Mixed values: includes negatives and zeros
x_mixed = np.random.normal(loc=1.5, scale=3.8, size=1200)
x_mixed[:5] = [-12, -8, -3, 0, 25]

df = pd.DataFrame({"Amount": x})
print("Sample rows:")
display(df.head())
print("\nAmount summary:")
display(df["Amount"].describe().to_frame().round(2))
print("Amount skewness:", round(skew(x), 3))

# PART A — FEATURE TRANSFORMATIONS

## 1. First understand the problem: what is skewness?

A **symmetric** distribution has roughly similar tails on both sides. A **right-skewed** distribution has a long tail to the right because a few values are much larger than most observations.

Examples can include income, house prices, hospital costs, and transaction amounts.

**Why might we care?**
- Extreme magnitudes can dominate plots and some model relationships.
- A strongly skewed feature may benefit from compression.
- Some statistical procedures assume approximately normal errors or benefit from stabilized variance. But **many ML models do not require every input feature to be normally distributed**.

**Say this to students:** “The goal is not always to force a bell-shaped curve. The goal is to choose a representation that fits the problem and helps the model.”

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x, bins=45, kde=True, ax=axes[0])
axes[0].set_title("Original amount: histogram")
axes[0].set_xlabel("Amount")
sns.boxplot(x=x, ax=axes[1])
axes[1].set_title("Original amount: boxplot")
axes[1].set_xlabel("Amount")
plt.tight_layout()
plt.show()

print("Skewness:", round(skew(x), 3), "(positive and large means right-skewed)")

## 2. Log transformation

### Formula
For strictly positive \(x\):

\[
x' = \ln(x)
\]

For data that includes zero but is non-negative, a common choice is:

\[
x' = \ln(1+x) \quad \text{implemented as } \texttt{np.log1p(x)}
\]

### Intuition
The log scale **compresses large values more strongly** than small values. For example, increasing from 10 to 100 is a tenfold change, and the log representation makes multiplicative differences easier to reason about.

### When is it useful?
- Strong right skew.
- Features spanning several orders of magnitude.
- Relationships that are multiplicative or approximately exponential.
- Interpreting relative changes rather than raw absolute differences.

### When is it NOT suitable?
- Ordinary \(\ln(x)\) cannot take zero or negative inputs.
- `log1p` handles zero and positive values, but not values below -1.
- It changes the interpretation and units of the feature.
- It is not a magic outlier remover.

**Say this to students:** “Log squeezes the long right tail. It does not delete large observations; it changes how far apart their values look.”

In [ ]:
x_log = np.log1p(x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x, bins=45, kde=True, ax=axes[0])
axes[0].set_title("BEFORE: original amount")
axes[0].set_xlabel("Amount")
sns.histplot(x_log, bins=45, kde=True, ax=axes[1])
axes[1].set_title("AFTER: log(1 + amount)")
axes[1].set_xlabel("log1p(amount)")
plt.tight_layout()
plt.show()

print(f"Skewness: {skew(x):.3f} before  ->  {skew(x_log):.3f} after")

In [ ]:
# See how selected values are compressed
demo_values = np.array([1, 5, 10, 50, 100, 250])
display(pd.DataFrame({
    "Original x": demo_values,
    "log(1 + x)": np.log1p(demo_values)
}).round(3))

**How to read the proof graph:** Compare the long tail and the skewness number. If the right tail becomes shorter and skewness decreases, log compression reduced right skewness for this example. It does **not** prove a model will perform better; validate that separately.

## 3. Square-root transformation

### Formula
For \(x \geq 0\):

\[
x' = \sqrt{x}
\]

### Intuition
Square root also compresses large values, but often less aggressively than a log transformation.

### When is it useful?
- Count-like features such as number of calls, visits, or defects.
- Moderate right skew where log may be too aggressive.
- Non-negative measurements where a gentler compression is reasonable.

### Limitations
- The ordinary real-valued square root is not defined for negative inputs.
- It may not be enough for very strong skew.
- The best choice depends on the distribution and domain.

**Say this to students:** “Square root is a gentler squeeze. Try it when the tail is moderately long, then compare the graph and model validation.”

In [ ]:
x_sqrt = np.sqrt(x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x, bins=45, kde=True, ax=axes[0])
axes[0].set_title("BEFORE: original amount")
axes[0].set_xlabel("Amount")
sns.histplot(x_sqrt, bins=45, kde=True, ax=axes[1])
axes[1].set_title("AFTER: square root")
axes[1].set_xlabel("sqrt(amount)")
plt.tight_layout()
plt.show()
print(f"Skewness: {skew(x):.3f} before  ->  {skew(x_sqrt):.3f} after")

## 4. Exponential transformation

### Formula
\[
x' = e^x
\]
More generally, \(x'=a^x\), where \(a>0\) and \(a\ne1\).

### Intuition
Exponentiation expands differences among larger input values. It is useful when the underlying process or model relationship is exponential, rather than as a routine way to reduce right skew.

### When might it be useful?
- Modeling a known exponential growth relationship.
- Re-expressing a variable when domain knowledge supports an exponential relationship.
- In specialized situations, transforming left-skewed values may help, but inspect the result carefully.

### Critical caution
**Exponentiation often makes right-skewed data more extreme.** It is generally not the first choice when the problem is a long right tail. We standardize the input for this demonstration to keep the exponentials numerically manageable.

**Say this to students:** “Not every transformation makes data better. Exponential transformation expands; log compresses. The graph tells us which effect happened.”

In [ ]:
# Standardize first for a stable demonstration, then exponentiate
x_centered = (x - x.mean()) / x.std()
x_exp = np.exp(x_centered)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x_centered, bins=45, kde=True, ax=axes[0])
axes[0].set_title("BEFORE: centered/scaled input")
axes[0].set_xlabel("(x - mean) / standard deviation")
sns.histplot(x_exp, bins=45, kde=True, ax=axes[1])
axes[1].set_title("AFTER: exp(standardized input)")
axes[1].set_xlabel("exp(z)")
plt.tight_layout()
plt.show()
print(f"Skewness: {skew(x_centered):.3f} before  ->  {skew(x_exp):.3f} after")

## 5. Box–Cox transformation

### Formula
For \(x>0\):

\[
x' =
\begin{cases}
\\dfrac{x^\\lambda - 1}{\\lambda}, & \\lambda \\ne 0 \\\\
\\ln(x), & \\lambda = 0
\end{cases}
\]

The parameter \(\lambda\) is estimated from the data. Different values behave like different power transformations; when \(\lambda=0\), the formula becomes log.

### When is it useful?
- Feature values are **strictly positive**.
- You want the algorithm to estimate a suitable power instead of choosing log or square root manually.
- You want to reduce skewness or stabilize variance.

### Limitations
- Strictly positive values only: zero and negative values are not allowed.
- It does not guarantee a normal distribution.
- Estimate preprocessing parameters from training data only.

**Say this to students:** “Box–Cox tries a family of curves and selects a lambda. Lambda is a learned setting, not a value we guess by eye.”

In [ ]:
x_boxcox, boxcox_lambda = boxcox(x)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x, bins=45, kde=True, ax=axes[0])
axes[0].set_title("BEFORE: original positive values")
axes[0].set_xlabel("Amount")
sns.histplot(x_boxcox, bins=45, kde=True, ax=axes[1])
axes[1].set_title(f"AFTER: Box–Cox (lambda={boxcox_lambda:.3f})")
axes[1].set_xlabel("Box–Cox value")
plt.tight_layout()
plt.show()
print("Estimated lambda:", round(boxcox_lambda, 3))
print(f"Skewness: {skew(x):.3f} before  ->  {skew(x_boxcox):.3f} after")

## 6. Power transformations: Box–Cox vs Yeo–Johnson

“Power transformation” is a family of transformations. In scikit-learn, `PowerTransformer` offers two common options:

- **Box–Cox:** requires strictly positive values.
- **Yeo–Johnson:** can handle negative values, zero, and positive values.

Both estimate a power parameter from the training data. By default, scikit-learn's `PowerTransformer` standardizes the transformed result; below we turn that off so the graph focuses on the power transformation itself.

### When should you use it?
- You want an automatic power-based approach to reduce skewness.
- Use Box–Cox for positive-only data.
- Use Yeo–Johnson if values include zero or negatives.

### Important
A lower skewness score is useful evidence about shape, but not a universal quality score. Check the graph, the domain, and downstream validation performance.

In [ ]:
# Yeo–Johnson accepts negative, zero, and positive values
yeo = PowerTransformer(method="yeo-johnson", standardize=False)
x_yj = yeo.fit_transform(x_mixed.reshape(-1, 1)).ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(x_mixed, bins=45, kde=True, ax=axes[0])
axes[0].set_title("BEFORE: mixed negative/zero/positive")
axes[0].set_xlabel("Original value")
sns.histplot(x_yj, bins=45, kde=True, ax=axes[1])
axes[1].set_title("AFTER: Yeo–Johnson")
axes[1].set_xlabel("Transformed value")
plt.tight_layout()
plt.show()
print("Estimated lambda:", round(yeo.lambdas_[0], 3))
print(f"Skewness: {skew(x_mixed):.3f} before  ->  {skew(x_yj):.3f} after")

## 7. Transformation comparison — same original feature

This is a fair visual comparison because each transformation starts from the same positive feature. The x-axis changes between panels, so compare **shape and tail behavior**, not the numerical positions as if units were identical.

In [ ]:
transformations = {
    "Original": x,
    "Log1p": np.log1p(x),
    "Square root": np.sqrt(x),
    "Box–Cox": x_boxcox
}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, values) in zip(axes.flat, transformations.items()):
    sns.histplot(values, bins=40, kde=True, ax=ax)
    ax.set_title(f"{name} | skew={skew(values):.2f}")
    ax.set_ylabel("Count")
plt.suptitle("One feature, different representations", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "Method": list(transformations.keys()),
    "Skewness": [skew(v) for v in transformations.values()],
    "Minimum": [np.min(v) for v in transformations.values()],
    "Maximum": [np.max(v) for v in transformations.values()]
}).round(3))

## 8. Transformation decision guide

Use this as a **starting point**, not an automatic rule.

| Data problem / scenario | First method to try | Why / caution |
|---|---|---|
| Strong right skew, positive values | Log or Box–Cox | Compresses large values; compare before/after |
| Moderate right skew, non-negative counts | Square root | Gentler compression |
| Positive values; want data-driven power choice | Box–Cox | Estimates lambda; requires values > 0 |
| Values include negatives or zeros | Yeo–Johnson | Supports mixed signs |
| Known exponential relationship | Exponential form may be appropriate | Often increases right skew; use domain reasoning |
| Feature already looks reasonable | Possibly no transformation | Unnecessary changes can hurt interpretation |
| Goal is just to put columns on comparable scales | Scaling, not transformation | Scaling does not specifically fix skewness |

**A practical rule:** first identify the problem, then choose a method. Do not apply log + square root + Box–Cox to the same feature one after another without a specific reason.

# PART B — FEATURE SCALING

## 9. Why scaling is needed

Imagine one row describes a person:

- Age = 25
- Salary = ₹80,000
- Experience = 2 years

The raw numerical size of salary is much larger. Algorithms based on distances or optimization may give salary disproportionate influence simply because of its units.

**Scaling** changes the numerical scale of features. It does not necessarily make a distribution normal or remove outliers.

Scaling often matters for:
- KNN and K-Means (distance-based)
- SVM
- PCA
- Logistic/linear regression with regularization
- Neural networks and many gradient-based methods

Tree-based models such as Decision Trees and Random Forests usually do not require feature scaling.

**Say this to students:** “A salary measured in rupees is numerically bigger than age. Scaling stops the unit size from automatically dominating distance or optimization.”

In [ ]:
# Features with different ranges and one deliberately large salary outlier
np.random.seed(7)
scale_df = pd.DataFrame({
    "Age": np.random.randint(18, 71, 300),
    "Salary": np.random.normal(65000, 18000, 300).clip(15000, 150000),
    "Experience": np.random.randint(0, 31, 300)
})
scale_df.loc[0, "Salary"] = 500000

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=scale_df, ax=axes[0])
axes[0].set_title("BEFORE: original feature scales")
axes[0].tick_params(axis="x", rotation=15)
sns.histplot(scale_df["Salary"], bins=35, kde=True, ax=axes[1])
axes[1].set_title("Salary includes an extreme value")
plt.tight_layout()
plt.show()
display(scale_df.describe().round(2))

## 10. StandardScaler (Z-score standardization)

### Formula
\[
z = \\frac{x-\\mu}{\\sigma}
\]

- \(\mu\) = mean of the feature.
- \(\sigma\) = standard deviation of the feature.

### What happens?
The fitted training feature is centered around 0 and has standard deviation 1 (using the scaler's population-standard-deviation convention).

### Best starting scenarios
- SVM, PCA, regularized linear/logistic regression, and many gradient-based models.
- Features with different units where outliers are not overwhelming.

### Limitation
Mean and standard deviation react to extreme values, so this scaler is sensitive to outliers.

In [ ]:
standard_scaler = StandardScaler()
standardized = pd.DataFrame(
    standard_scaler.fit_transform(scale_df), columns=scale_df.columns
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=scale_df, ax=axes[0])
axes[0].set_title("BEFORE: original values")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=standardized, ax=axes[1])
axes[1].set_title("AFTER: StandardScaler")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

print("Means after scaling:")
display(standardized.mean().to_frame("Mean").round(3))
print("Standard deviations after scaling:")
display(standardized.std(ddof=0).to_frame("Std dev").round(3))

**What this graph proves:** the columns now use a comparable standardized scale. The salary outlier still has influence — StandardScaler does not remove outliers.

## 11. MinMaxScaler

### Formula
\[
x' = \\frac{x-x_{\\min}}{x_{\\max}-x_{\\min}}
\]

With the default range, the observed minimum maps to 0 and the observed maximum maps to 1.

### Best starting scenarios
- A model or workflow benefits from features in a fixed range.
- Neural-network inputs where a bounded input range is useful.
- You want to preserve relative ordering while rescaling the range.

### Limitation
Very sensitive to outliers: a huge maximum can squeeze most other observations into a narrow interval. New future data can also fall outside 0–1 when transformed with the training-fitted scaler.

In [ ]:
minmax_scaler = MinMaxScaler()
minmax = pd.DataFrame(minmax_scaler.fit_transform(scale_df), columns=scale_df.columns)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=scale_df, ax=axes[0])
axes[0].set_title("BEFORE: original values")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=minmax, ax=axes[1])
axes[1].set_title("AFTER: MinMaxScaler (training sample 0–1)")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"Minimum": minmax.min(), "Maximum": minmax.max()}).round(3))

## 12. RobustScaler

### Formula
\[
x' = \\frac{x-\\operatorname{median}(x)}{Q_3-Q_1}
\]

- Median = 50th percentile.
- \(Q_1\) = 25th percentile.
- \(Q_3\) = 75th percentile.
- IQR = \(Q_3-Q_1\).

### Best starting scenarios
- Numeric features contain strong outliers.
- You want centering and scaling based on statistics that are less affected by extreme values.

### Limitation
It does not delete outliers or cap their transformed values. It simply uses median and IQR to calculate the scale.

In [ ]:
robust_scaler = RobustScaler()
robust = pd.DataFrame(robust_scaler.fit_transform(scale_df), columns=scale_df.columns)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=scale_df, ax=axes[0])
axes[0].set_title("BEFORE: original values")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=robust, ax=axes[1])
axes[1].set_title("AFTER: RobustScaler")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()
print("Medians after RobustScaler:")
display(robust.median().to_frame("Median").round(3))

## 13. MaxAbsScaler

### Formula
\[
x' = \\frac{x}{\\max(|x|)}
\]

Each feature is divided by its maximum absolute value, so the training values are within \([-1,1]\).

### Best starting scenarios
- Sparse features where preserving zeros matters.
- Data where you want to scale by maximum magnitude without centering.

### Limitation
Like Min–Max scaling, it is sensitive to extreme values because the maximum absolute value controls the scale.

In [ ]:
maxabs_scaler = MaxAbsScaler()
maxabs = pd.DataFrame(maxabs_scaler.fit_transform(scale_df), columns=scale_df.columns)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=scale_df, ax=axes[0])
axes[0].set_title("BEFORE: original values")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=maxabs, ax=axes[1])
axes[1].set_title("AFTER: MaxAbsScaler")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()
display(pd.DataFrame({"Minimum": maxabs.min(), "Maximum": maxabs.max()}).round(3))

## 14. Normalizer — different from the scalers above

### L2 formula
For a sample vector \(\mathbf{x}\):

\[
\\mathbf{x}' = \\frac{\\mathbf{x}}{\\|\\mathbf{x}\\|_2},
\\quad
\\|\\mathbf{x}\\|_2=\\sqrt{x_1^2+x_2^2+\\cdots+x_n^2}
\]

**Key distinction:** StandardScaler, MinMaxScaler, RobustScaler, and MaxAbsScaler usually scale each **column**. `Normalizer` scales each **row/sample** to unit length.

### Best starting scenarios
- Text vectors, document similarity, cosine similarity, and cases where vector direction matters more than magnitude.

### Not a general replacement
For ordinary tabular data such as age/salary/experience, row-wise normalization may change the meaning of each person's values in an undesirable way.

In [ ]:
normalizer = Normalizer(norm="l2")
normalized_rows = pd.DataFrame(normalizer.fit_transform(scale_df), columns=scale_df.columns)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=scale_df, ax=axes[0])
axes[0].set_title("BEFORE: original feature scales")
axes[0].tick_params(axis="x", rotation=15)
sns.boxplot(data=normalized_rows, ax=axes[1])
axes[1].set_title("AFTER: row-wise L2 normalization")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()
print("First 5 row L2 norms:", np.round(np.linalg.norm(normalized_rows.to_numpy(), axis=1)[:5], 4))

## 15. Scaling comparison: inspect the same features side by side

Each panel shows the same columns after a different method. This makes the methods' behavior easier to compare.

**Teaching prompt:** “Which scaler seems less affected by the very large salary? Which scaler forces the values into a fixed range? Which one centers around zero?”

In [ ]:
scaled_versions = {
    "StandardScaler": standardized,
    "MinMaxScaler": minmax,
    "RobustScaler": robust,
    "MaxAbsScaler": maxabs
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (name, data) in zip(axes.flat, scaled_versions.items()):
    melted = data.melt(var_name="Feature", value_name="Scaled value")
    sns.boxplot(data=melted, x="Feature", y="Scaled value", ax=ax)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 16. Scaling decision guide — which is best for which scenario?

| Scenario | Good first choice | Why |
|---|---|---|
| SVM, PCA, regularized linear/logistic regression | StandardScaler | Centers features and expresses them in standard-deviation units |
| Need a fixed feature range, often 0–1 | MinMaxScaler | Maps fitted training minimum/maximum to the chosen range |
| Strong numeric outliers | RobustScaler | Uses median and IQR rather than mean and standard deviation |
| Sparse data; preserve zeros and avoid centering | MaxAbsScaler | Divides by maximum absolute value |
| Text/document vectors or direction-based similarity | Normalizer | Makes each row have unit norm |
| Decision tree / Random Forest | Often no scaling needed | Split decisions are generally insensitive to monotonic scale changes |
| Strong skewness is the issue | Consider a transformation first | Scaling alone does not usually fix skewness |

**Best means “best for the problem,” not “best in every dataset.”** Compare methods using validation data and the actual model metric.

# PART C — PUTTING IT TOGETHER

## 17. Transformation vs scaling: do not confuse them

| Question | Transformation | Scaling |
|---|---|---|
| Main purpose | Change distribution shape or relationship | Put feature magnitudes on a comparable scale |
| Example | Log1p, square root, Box–Cox, Yeo–Johnson | StandardScaler, MinMaxScaler, RobustScaler |
| Can reduce right skew? | Often, depending on method and data | Usually not its purpose |
| Can fix different measurement units? | Not necessarily | Yes, often |
| Does it remove outliers? | No, although some compress their magnitude | No |
| Does every model need it? | No | No |

### A practical decision sequence
1. **Plot the feature.** Is it strongly skewed? Are there impossible values or outliers?
2. **Choose a transformation only if justified.** For right-skewed positive data, try log1p or Box–Cox; for mixed signs, consider Yeo–Johnson.
3. **Inspect the result.** Compare plots and summary statistics.
4. **Scale if the algorithm needs comparable feature scales.**
5. **Validate the complete pipeline.** Better-looking plots do not guarantee better predictions.

## 18. Important: prevent data leakage

Never fit a scaler or learn a transformation parameter on the entire dataset before splitting. That allows information from the test set to influence preprocessing.

### Correct order
1. Split into training and test sets.
2. Fit preprocessing on training data only.
3. Transform training data and test data using the fitted training transformer.
4. Train and evaluate the model.

Use a pipeline so preprocessing is refitted correctly inside cross-validation.

```python
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)
print("Test accuracy:", pipe.score(X_test, y_test))
```

Here `X` and `y` represent your real feature matrix and target column; define them for your dataset before running this example.

## 19. Classroom recap: one-minute memory map

- **Log:** compresses a long right tail strongly; positive values, or use `log1p` for non-negative values.
- **Square root:** gentler compression for non-negative values/counts.
- **Exponential:** expands differences; use when the relationship calls for it, not as a routine right-skew fix.
- **Box–Cox:** estimates a power for strictly positive data.
- **Yeo–Johnson:** estimates a power and supports negative, zero, and positive values.
- **StandardScaler:** mean 0, standard deviation 1; sensitive to outliers.
- **MinMaxScaler:** fixed range; sensitive to outliers.
- **RobustScaler:** median/IQR; a sensible first try with outliers.
- **MaxAbsScaler:** preserves zeros; useful for sparse features.
- **Normalizer:** scales rows, often for vectors and similarity.

### Exit questions for students
1. If salary has a long right tail, which transformations could you compare?
2. If a feature has negative values, why might ordinary Box–Cox fail?
3. If a dataset has extreme outliers, which scaler is worth trying first?
4. Why does MinMaxScaler not guarantee future values stay between 0 and 1?
5. Why should we fit preprocessing on training data only?
6. If the graph becomes more bell-shaped, does that prove model accuracy improves? Why not?

**Final lesson:** Choose preprocessing to solve a diagnosed problem, prove its effect with graphs and statistics, and judge predictive value using leakage-safe validation.